In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, average_precision_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline



In [2]:
df = pd.read_csv("dados_usados/dados_para_modelo.csv")

In [3]:
df.head(10)

,status_do_emprestimo,valor_do_emprestimo,prazo_do_emprestimo_meses,taxa_de_juros,valor_da_parcela,tempo_de_emprego_anos,renda_anual,data_de_emissao_do_emprestimo,relacao_divida_renda,primeira_linha_de_credito,...,total_de_contas_de_credito,contas_de_hipoteca,registros_de_falencia,classificacao_de_risco,tipo_de_moradia,finalidade_do_emprestimo,tipo_de_aplicacao,taxa_de_juros_agrupada,taxa/falencia,juros/contas
0,0,10000.0,36,11.44,329.48,10.0,117000.0,2015,26.24,1990,...,25.0,0.0,0.0,0.874270,0.773378,0.810767,0.803913,0.25,11.44,64.0
1,0,8000.0,36,11.99,265.68,4.0,65000.0,2015,22.05,2004,...,27.0,3.0,0.0,0.874270,0.830439,0.792586,0.803913,0.50,11.99,34.0
2,0,15600.0,36,10.49,506.97,0.0,43057.0,2015,12.79,2007,...,26.0,0.0,0.0,0.874270,0.773378,0.832882,0.803913,0.25,10.49,52.0
3,0,7200.0,36,6.49,220.65,6.0,54000.0,2014,2.60,2006,...,13.0,0.0,0.0,0.937121,0.773378,0.832882,0.803913,0.25,6.49,24.0
4,1,24375.0,60,17.27,609.33,9.0,55000.0,2013,33.95,1999,...,43.0,1.0,0.0,0.788191,0.830439,0.832882,0.803913,0.50,17.27,26.0
5,0,20000.0,36,13.33,677.07,10.0,86788.0,2015,16.31,2005,...,23.0,4.0,0.0,0.788191,0.830439,0.792586,0.803913,0.50,13.33,16.0
6,0,18000.0,36,5.32,542.07,2.0,125000.0,2015,1.36,2005,...,25.0,3.0,0.0,0.937121,0.830439,0.829921,0.803913,0.25,5.32,32.0
7,0,13000.0,36,11.14,426.47,10.0,46000.0,2012,26.87,1994,...,15.0,0.0,0.0,0.874270,0.773378,0.832882,0.803913,0.25,11.14,44.0
8,0,18900.0,60,10.99,410.84,10.0,103000.0,2014,12.52,1994,...,40.0,3.0,0.0,0.874270,0.773378,0.792586,0.803913,0.25,10.99,52.0
9,0,26300.0,36,16.29,928.40,3.0,115000.0,2012,23.69,1997,...,37.0,1.0,0.0,0.788191,0.830439,0.792586,0.803913,0.50,16.29,26.0


In [4]:
x = df.drop("status_do_emprestimo", axis=1)
y = df["status_do_emprestimo"]

In [5]:
x.info()

<class 'pandas.DataFrame'>
RangeIndex: 383421 entries, 0 to 383420
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   valor_do_emprestimo             383421 non-null  float64
 1   prazo_do_emprestimo_meses       383421 non-null  int64  
 2   taxa_de_juros                   383421 non-null  float64
 3   valor_da_parcela                383421 non-null  float64
 4   tempo_de_emprego_anos           383421 non-null  float64
 5   renda_anual                     383421 non-null  float64
 6   data_de_emissao_do_emprestimo   383421 non-null  int64  
 7   relacao_divida_renda            383421 non-null  float64
 8   primeira_linha_de_credito       383421 non-null  int64  
 9   contas_de_credito_abertas       383421 non-null  float64
 10  registros_publicos_negativos    383421 non-null  float64
 11  saldo_de_credito_rotativo       383421 non-null  float64
 12  utilizacao_do_credito_rotat

In [6]:
y.value_counts()

status_do_emprestimo
0    308372
1     75049
Name: count, dtype: int64

In [7]:
x_train, x_test, y_train, y_test= train_test_split(x, y, test_size=0.2, stratify= y, random_state=42)

In [8]:
len(x_train), len(y_train)

(306736, 306736)

In [9]:
len(x_test), len(y_test)

(76685, 76685)

In [10]:
pipeline_logistic = Pipeline(
   [ ("scaler", StandardScaler()),
    ("model", LogisticRegression())]
)

pipeline_knn = Pipeline(
    [("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())]
)

In [11]:
modelos = {
    "KNN" : pipeline_knn,
    "Logistic Regressionr" : pipeline_logistic,
    "Ramdom Forest" : RandomForestClassifier(random_state=42),
    "XGBoosting" : XGBClassifier(random_state=42),
    "Lightgbm" : LGBMClassifier(random_state=42)
}

In [ ]:
avaliacao = {}

for name, modelo in modelos.items():
    cross_score = cross_val_score(
        modelo, x_train, y_train,
        cv=5,
        scoring= "roc_auc"
    )
    avaliacao[name] = np.mean(cross_score)
    print(f"{name} : {avaliacao[name]}")


' valiacao = {}\n\nfor name, modelo in modelos.items():\n    cross_score = cross_val_score(\n        modelo, x_train, y_train,\n        cv=5,\n        scoring= "roc_auc"\n    )\n    avaliacao[name] = np.mean(cross_score)\n    print(f"{name} : {avaliacao[name]}")'

In [15]:
knn_params = {
  "model__n_neighbors" : [1, 3, 5, 7, 9, 20, 30, 50],
  "model__weights" : ["uniform", "distance"]
}

random_forest_params = {
    "n_estimators" : np.arange(300, 1500, 90),
    "max_depth" : [1, 5, 9, 20, 50],
    "min_samples_leaf" : np.arange(1, 10, 3),
    "min_samples_split" : np.arange(1, 15, 3)
}

logistic_params = {
    "model__C" : np.logspace(-3, 3, 10),
    "model__solver" : ["lbfgs", "liblinear"]
}

gradient_boosting_params = {
    "n_estimators" : np.arange(100, 1080, 80),
    "max_depth" : [1, 3, 9, 15],
    "subsample" : [0.2, 0.5, 0.7, 1],
    "min_child_weight" : np.arange(1, 10, 3),
    "scale_pos_weight" : [10, 20, 5, 15, ]
}

light_params = {
    "num_leaves" : [63, 127],
    "n_estimators" : [100, 300, 500, 1000],
    "min_child_sample" : [100, 300, 500, 1000],
    "max_depth" : [1, 5, 7],
    "subsample" : [0.2, 0.5, 1],
    "is_unbalance" : [True],
    "colsample_bytree" : [0.2, 1],
    "learning_rate" : [0.2, 1],
    "force_row_wise" : [True]
}



In [132]:
x_sample,_, y_sample,_ = train_test_split(x_train, y_train, train_size=100000, stratify=y_train, random_state=42)

In [13]:
def test_randomized_search(model, params, x_sample, y_sample):
    randomized = HalvingRandomSearchCV(
    model,
    params,
    cv= 5,
    n_candidates = 50,
    factor=3,
    scoring="roc_auc",
    n_jobs = -1,
    min_resources= 6000,
    random_state=42,
    verbose=2
)

    randomized.fit(x_sample, y_sample)
    return randomized
    print(f"O melhor score foi: {randomized.best_score_}")
    print(f"Os melhores parametros foram: {randomized.best_params_}")


In [134]:
knn_modelo = test_randomized_search(pipeline_knn, knn_params, x_train, y_train)

n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 4
min_resources_: 6000
max_resources_: 306736
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 16
n_resources: 6000
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/home/Erick/miniconda3/envs/data/lib/python3.11/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 16 is smaller than n_iter=50. Running 16 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.2s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.1s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.1s
[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.2s
[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.2s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.2s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.2s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.2s
[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.1s
[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.1s
[CV] END .......model__n_neighbors=3, model__weights=uniform; total time=   0.1s
[CV] END .......model__n_neighbors=3, model__weights=uniform; total time=   0.1s
[CV] END .......model__n_nei

In [135]:
logistic_modelo = test_randomized_search(pipeline_logistic, logistic_params, x_train, y_train)

n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 4
min_resources_: 6000
max_resources_: 306736
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 20
n_resources: 6000
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/home/Erick/miniconda3/envs/data/lib/python3.11/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 20 is smaller than n_iter=50. Running 20 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.1s
[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.1s
[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.1s
[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.1s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.1s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.1s[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.1s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.1s

[CV] END .model__C=0.004641588833612777, model__solver=lbfgs; total time=   0.1s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.1s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.1s
[CV] END .model__C=0.004641588833612777, model__solver=lbfgs; total time=   0.1s
[CV] END .model__C=0.0046415

In [16]:
xgb_modelo = test_randomized_search(XGBClassifier(), gradient_boosting_params, x_train, y_train)

n_iterations: 4
n_required_iterations: 4
n_possible_iterations: 4
min_resources_: 6000
max_resources_: 306736
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 50
n_resources: 6000
Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END max_depth=15, min_child_weight=7, n_estimators=100, scale_pos_weight=10, subsample=1; total time=   2.6s
[CV] END max_depth=15, min_child_weight=7, n_estimators=100, scale_pos_weight=10, subsample=1; total time=   2.6s
[CV] END max_depth=15, min_child_weight=7, n_estimators=100, scale_pos_weight=10, subsample=1; total time=   2.6s
[CV] END max_depth=15, min_child_weight=7, n_estimators=100, scale_pos_weight=10, subsample=1; total time=   2.7s
[CV] END max_depth=15, min_child_weight=7, n_estimators=100, scale_pos_weight=10, subsample=1; total time=   2.7s
[CV] END max_depth=3, min_child_weight=4, n_estimators=420, scale_pos_weight=20, subsample=0.7; total time=   0.7s
[CV] END max_depth=3, min_child_weight=4, n_estim

In [137]:
#lightgbm testando os hiperparametros estava travando meu pc, então não pude testar, 
# mas mesmo testando hiperparametros dos outros modelos, 
# ele ainda é um pouco superior mesmo sem ajustar os hiperparametros
#light_modelo = test_randomized_search(LGBMClassifier(), light_params, x_sample, y_sample)

In [138]:
#random_forest_modelo = test_randomized_search(RandomForestClassifier(), random_forest_params, x_sample, y_sample)

In [140]:
best_modelo = XGBClassifier(subsample= 1, scale_pos_weight= 10, n_estimators= np.int64(340), min_child_weight =np.int64(4), max_depth =1, random_state= 42)

In [141]:
best_modelo.fit(x_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [142]:
y_proba = best_modelo.predict_proba(x_test)[:, 1]

In [143]:
roc_auc = roc_auc_score(y_test, y_proba)
roc_auc

0.720239383715815

In [134]:
avaliacao_certa = 0
melhor_avaliacao = []
avaliacao = np.arange(0.4, 0.95, 0.04)

In [144]:
for i in avaliacao:
    y_pred = (y_proba >= i).astype(int)
    melhor_avaliacao = f1_score(y_test, y_pred)
    if melhor_avaliacao > avaliacao_certa:
        avaliacao_certa = melhor_avaliacao
melhor_avaliacao


0.07349375116582726

In [145]:
y_pred = (y_proba >= 0.6959375116582726).astype(int)

In [138]:
f1 = f1_score(y_test, y_pred)
f1

0.4312485888462407

In [146]:
precision = precision_score(y_test, y_pred)
precision

0.31178267445653784

In [147]:
recall = recall_score(y_test, y_pred)
recall

0.7003997335109927

In [148]:
average = average_precision_score(y_test, y_proba)
average

0.38051914530471975

In [149]:
confusion_matrix(y_test, y_pred)

array([[38469, 23206],
       [ 4497, 10513]])